<a href="https://colab.research.google.com/github/Shiblu31/ML_Lab_R/blob/main/Lab_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Load dataset
df = pd.read_csv("/content/drive/MyDrive/DATASET/Titanic-Dataset.csv")

print(df.head())
print(df.info())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
<c

Step 2: Data Cleaning

In [6]:
#Handle Missing Values
# Fill Age with median
df['Age'].fillna(df['Age'].median(), inplace=True)

# Fill Embarked with mode
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# Drop Cabin (too many missing values)
df.drop(columns=['Cabin'], inplace=True)

# Drop duplicates
df.drop_duplicates(inplace=True)

/tmp/ipykernel_1282/3918048445.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
/tmp/ipykernel_1282/3918048445.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', tr

Step 3: Data Transformation (Encoding)

In [7]:
# Convert categorical to numerical

# Sex
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# Embarked (One-hot encoding)
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

Step 4: Feature Engineering


In [8]:
# Create Family Size
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Create IsAlone feature
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Create Age Group
df['AgeGroup'] = pd.cut(df['Age'],
                       bins=[0, 12, 18, 35, 60, 100],
                       labels=[0,1,2,3,4])

Step 5: Feature Selection

In [9]:
X = df.drop(columns=['Survived', 'Name', 'Ticket', 'PassengerId'])
y = df['Survived']

In [10]:
#Filter Method (Chi-Square)
selector = SelectKBest(score_func=chi2, k=5)
X_new = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]
print("Selected features (Chi2):", selected_features)

Selected features (Chi2): Index(['Pclass', 'Sex', 'Age', 'Fare', 'IsAlone'], dtype='object')


In [11]:
model = RandomForestClassifier()
model.fit(X, y)

importance = pd.Series(model.feature_importances_, index=X.columns)
print("Feature Importance:\n", importance.sort_values(ascending=False))

Feature Importance:
 Sex           0.261648
Fare          0.255530
Age           0.212826
Pclass        0.086913
FamilySize    0.044828
AgeGroup      0.043671
SibSp         0.028866
Parch         0.023426
Embarked_S    0.022040
IsAlone       0.010275
Embarked_Q    0.009977
dtype: float64


Step 6: Feature Scaling

In [12]:
#Standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [13]:
#normalizatin
norm = MinMaxScaler()
X_normalized = norm.fit_transform(X)

Step 7: Model Training (Before vs After)

In [14]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print("Accuracy after preprocessing:", accuracy)

Accuracy after preprocessing: 0.7988826815642458
